# Class OI Analitycs

In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from urllib.parse import quote
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats


class DataProcessor:
    """Classe pour gérer la récupération et le traitement de données depuis une API."""
    
    def __init__(self, url_base, tags_other, tags_selected, start, end, 
                 interval='PT20M', hS='00', hF='23', cred_file="../../../cred.txt"):
        """
        Initialise le processeur de données.
        
        Parameters:
        url_base : str, URL de base de l'API
        tags_other : list, liste des tags autres (non renommés)
        tags_selected : list of dict, liste des tags sélectionnés avec renommage
                        Format: [{'tag': 'nom_api', 'nom': 'nom_final'}, ...]
        start : str, date de début (format 'YYYY-MM-DD')
        end : str, date de fin (format 'YYYY-MM-DD')
        interval : str, intervalle d'agrégation (défaut: 'PT20M')
        hS : str, heure de début (défaut: '00')
        hF : str, heure de fin (défaut: '23')
        cred_file : str, chemin vers le fichier de credentials
        """
        self.url_base = url_base
        self.tags_other = tags_other
        self.tags_selected = tags_selected
        self.start = start
        self.end = end
        self.interval = interval
        self.hS = hS
        self.hF = hF
        self.credentials = self._read_cred(cred_file)
        
        # Dictionnaire de mapping pour le renommage
        self.rename_mapping = {item['tag']: item['nom'] for item in tags_selected}
        
        # DataFrames
        self.df = None  # DataFrame brut fusionné (avec noms API)
        self.data = None  # Copie de travail (avec noms finaux)
        
    def _read_cred(self, cred_file):
        """
        Lit les credentials depuis un fichier.
        
        Parameters:
        cred_file : str, chemin vers le fichier
        
        Returns:
        str : contenu du fichier credentials
        """
        with open(cred_file, "r") as f:
            cred = f.read()
        return cred
    
    def _get_all_tags_api(self):
        """
        Retourne tous les tags pour les requêtes API (noms originaux).
        
        Returns:
        list : liste de tous les tags API
        """
        tags_selected_api = [item['tag'] for item in self.tags_selected]
        return self.tags_other + tags_selected_api
    
    def get_OI(self, tag):
        """
        Récupère les données depuis l'API pour un tag donné.
        
        Parameters:
        tag : str, tag à récupérer
        
        Returns:
        list : valeurs récupérées
        """
        url_all = (f"{self.url_base}data-reference={tag}&aggregation=TIME"
                   f"&aggregation-function=MEAN&from={self.start}T{self.hS}%3A00%3A00.000Z"
                   f"&to={self.end}T{self.hF}%3A59%3A59.000Z&aggregation-period={self.interval}")
        
        d_data = pd.read_json(url_all, storage_options={'Authorization': 'basic ' + self.credentials})
        arr = np.asarray(np.asarray(d_data['values'])[0])
        return d_data['values'][0]
    
    def get_data(self):
        """
        Récupère les données pour tous les tags.
        
        Returns:
        list : liste de DataFrames
        """
        tags = self._get_all_tags_api()
        liste = []
        for tag in tags:
            urlTag = quote(tag, safe=':/?#[]@!$&\'()*+,;=')
            data = self.get_OI(urlTag)
            df_temp = pd.DataFrame(data)
            df_temp['timestamp'] = pd.to_datetime(df_temp['timestamp'])
            df_temp = df_temp.set_index('timestamp')
            df_temp = df_temp.rename(columns={'value': tag})
            liste.append(df_temp)
        return liste
    
    def _rename_columns(self, df):
        """
        Renomme les colonnes du DataFrame selon le mapping défini.
        
        Parameters:
        df : DataFrame à renommer
        
        Returns:
        DataFrame avec colonnes renommées
        """
        return df.rename(columns=self.rename_mapping)
    
    def merge(self):
        """
        Fusionne les données récupérées dans self.df.
        Crée également une copie de travail dans self.data avec les colonnes renommées.
        """
        df_list = self.get_data()
        self.df = reduce(lambda left, right: pd.merge(left, right, left_index=True, 
                                                       right_index=True, how='outer'), df_list)
        # Créer data avec les colonnes renommées
        self.data = self._rename_columns(self.df.copy())
        print(f"Données chargées : {len(self.df)} lignes, {len(self.df.columns)} colonnes")
        print(f"Colonnes renommées : {list(self.rename_mapping.values())}")
    
    def read(self, start=None, end=None, interval=None, hS=None, hF=None):
        """
        Réinitialise et recharge le DataFrame avec de nouveaux paramètres.
        
        Parameters:
        start : str, date de début (optionnel)
        end : str, date de fin (optionnel)
        interval : str, intervalle d'agrégation (optionnel)
        hS : str, heure de début (optionnel)
        hF : str, heure de fin (optionnel)
        """
        # Mise à jour des paramètres si fournis
        if start is not None:
            self.start = start
        if end is not None:
            self.end = end
        if interval is not None:
            self.interval = interval
        if hS is not None:
            self.hS = hS
        if hF is not None:
            self.hF = hF
        
        # Réinitialisation et rechargement
        self.df = None
        self.data = None
        self.merge()
    
    def filtering(self, tag, min_val, max_val, na=None):
        """
        Applique des filtres sur self.data.
        IMPORTANT: Utiliser les noms finaux (renommés) des colonnes, pas les noms API.
        
        Parameters:
        tag : str or list, tag(s) à filtrer (noms finaux)
        min_val : float or list, valeur(s) minimale(s)
        max_val : float or list, valeur(s) maximale(s)
        na : str or list or None, tag(s) sur lequel appliquer dropna (noms finaux)
        
        Returns:
        DataProcessor : self pour chaînage
        """
        # Supprime les lignes entièrement vides
        self.data = self.data.dropna(how="all")
        
        # Convertit tag en liste si c'est une chaîne
        if isinstance(tag, str):
            tags = [tag]
            min_vals = [min_val]
            max_vals = [max_val]
        else:
            tags = tag
            min_vals = min_val if isinstance(min_val, list) else [min_val] * len(tags)
            max_vals = max_val if isinstance(max_val, list) else [max_val] * len(tags)
        
        # Applique les filtres min/max pour chaque tag
        for t, min_v, max_v in zip(tags, min_vals, max_vals):
            if t in self.data.columns:
                self.data = self.data[(self.data[t] > min_v) & (self.data[t] < max_v)]
            else:
                print(f"Attention : colonne '{t}' introuvable dans data")
        
        # Applique dropna si spécifié
        if na is not None:
            na_list = [na] if isinstance(na, str) else na
            self.data = self.data.dropna(how="all", subset=na_list)
        
        print(f"Après filtrage : {len(self.data)} lignes")
        return self
    
    def ajoute_cumul(self, col_poids, col_valeur, ratio, nom):
        """
        Ajoute une colonne de cumul à self.data.
        IMPORTANT: Utiliser les noms finaux (renommés) des colonnes.
        
        Parameters:
        col_poids : str, nom de la colonne de poids (nom final)
        col_valeur : str, nom de la colonne de valeur (nom final)
        ratio : float, ratio de division
        nom : str, nom de la nouvelle colonne
        
        Returns:
        DataProcessor : self pour chaînage
        """
        if col_poids in self.data.columns and col_valeur in self.data.columns:
            self.data[nom] = (self.data[col_poids] * self.data[col_valeur]) / ratio
            print(f"Colonne '{nom}' ajoutée")
        else:
            print(f"Erreur : colonnes '{col_poids}' ou '{col_valeur}' introuvables")
        return self
    
    def ajouter_moyennes_glissantes(self, col_poids, col_valeur, nom, window=10):
        """
        Ajoute une colonne de moyenne pondérée glissante à self.data.
        IMPORTANT: Utiliser les noms finaux (renommés) des colonnes.
        
        Parameters:
        col_poids : str, nom de la colonne de poids (nom final)
        col_valeur : str, nom de la colonne de valeurs (nom final)
        nom : str, nom de la nouvelle colonne
        window : int, taille de la fenêtre glissante
        
        Returns:
        DataProcessor : self pour chaînage
        """
        if col_poids in self.data.columns and col_valeur in self.data.columns:
            poids = self.data[col_poids]
            valeurs = self.data[col_valeur]
            
            numerateur = (poids * valeurs).rolling(window=window).sum()
            denominateur = poids.rolling(window=window).sum()
            self.data[nom] = numerateur / denominateur
            
            print(f"Colonne '{nom}' ajoutée (moyenne glissante sur {window} valeurs)")
        else:
            print(f"Erreur : colonnes '{col_poids}' ou '{col_valeur}' introuvables")
        return self
    
    def plot_tag(self, tag):
        """
        Affiche les graphiques temporel et de distribution pour un ou plusieurs tags.
        IMPORTANT: Utiliser le(s) nom(s) final(aux) (renommé(s)) de(s) colonne(s).
        
        Parameters:
        tag : str or list, nom du/des tag(s) à afficher (nom(s) final(aux))
        """
        if self.data is None:
            print("Erreur : Aucune donnée disponible. Exécutez merge() d'abord.")
            return
        
        # Convertir en liste si c'est une chaîne
        tags = [tag] if isinstance(tag, str) else tag
        
        # Vérifier que tous les tags existent
        tags_valides = []
        for t in tags:
            if t not in self.data.columns:
                print(f"Attention : colonne '{t}' introuvable dans data")
            else:
                tags_valides.append(t)
        
        if not tags_valides:
            print(f"Erreur : Aucun tag valide trouvé")
            print(f"Colonnes disponibles : {list(self.data.columns)}")
            return
        
        # Couleurs pour différencier les tags
        colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'cyan', 'magenta']
        
        for idx, tag_name in enumerate(tags_valides):
            color = colors[idx % len(colors)]
            
            # Calcul des statistiques
            data_series = self.data[tag_name].dropna()
            if len(data_series) == 0:
                print(f"Attention : tag '{tag_name}' ne contient aucune donnée valide")
                continue
                
            moyenne = data_series.mean()
            mediane = data_series.median()
            ecart_type = data_series.std()
            nb_valeurs = len(data_series)
            percentile_25 = data_series.quantile(0.25)
            percentile_75 = data_series.quantile(0.75)
            val_min = data_series.min()
            val_max = data_series.max()
            
            # Création de la figure avec 2 subplots
            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=(f'{tag_name} - Évolution temporelle', f'{tag_name} - Distribution'),
                horizontal_spacing=0.12,
                column_widths=[0.65, 0.35]
            )
            
            # === SUBPLOT 1 : Scatter plot ===
            # Points de données
            fig.add_trace(
                go.Scatter(
                    x=self.data.index,
                    y=self.data[tag_name],
                    mode='markers',
                    name=tag_name,
                    marker=dict(color=color, opacity=0.3, size=6),
                    hovertemplate='Date: %{x}<br>Valeur: %{y:.2f}<extra></extra>'
                ),
                row=1, col=1
            )
            
            # Ligne de moyenne
            fig.add_trace(
                go.Scatter(
                    x=self.data.index,
                    y=[moyenne] * len(self.data.index),
                    mode='lines',
                    name=f'Moyenne: {moyenne:.2f}',
                    line=dict(color='red', dash='dash', width=2),
                    hovertemplate=f'Moyenne: {moyenne:.2f}<extra></extra>'
                ),
                row=1, col=1
            )
            
            # Ligne de médiane
            fig.add_trace(
                go.Scatter(
                    x=self.data.index,
                    y=[mediane] * len(self.data.index),
                    mode='lines',
                    name=f'Médiane: {mediane:.2f}',
                    line=dict(color='violet', dash='dash', width=2),
                    hovertemplate=f'Médiane: {mediane:.2f}<extra></extra>'
                ),
                row=1, col=1
            )
            
            # Courbe de tendance
            x = np.arange(len(self.data.index))
            y = self.data[tag_name].values
            
            # Filtrer les valeurs NaN
            mask = ~np.isnan(y)
            x_clean = x[mask]
            y_clean = y[mask]
            
            if len(x_clean) > 1:
                # Régression linéaire
                coeffs = np.polyfit(x_clean, y_clean, 1)
                line = np.polyval(coeffs, x)
                
                # Calcul du R²
                y_pred = np.polyval(coeffs, x_clean)
                ss_res = np.sum((y_clean - y_pred) ** 2)
                ss_tot = np.sum((y_clean - np.mean(y_clean)) ** 2)
                r_squared = 1 - (ss_res / ss_tot)
                
                # Tracer la ligne de tendance
                fig.add_trace(
                    go.Scatter(
                        x=self.data.index,
                        y=line,
                        mode='lines',
                        name=f'Tendance (R²={r_squared:.3f})',
                        line=dict(color='green', width=2),
                        hovertemplate=f'Tendance (R²={r_squared:.3f})<br>Valeur: %{{y:.2f}}<extra></extra>'
                    ),
                    row=1, col=1
                )
            
            # === SUBPLOT 2 : Histogramme avec KDE ===
            # Histogramme
            fig.add_trace(
                go.Histogram(
                    x=data_series,
                    name='Distribution',
                    marker=dict(color='lightblue', line=dict(color='darkblue', width=1)),
                    opacity=0.7,
                    histnorm='probability density',
                    hovertemplate='Valeur: %{x:.2f}<br>Densité: %{y:.4f}<extra></extra>',
                    showlegend=False
                ),
                row=1, col=2
            )
            
            # Approximation KDE (Kernel Density Estimation)
            if len(data_series) > 1:
                kde = stats.gaussian_kde(data_series)
                x_range = np.linspace(data_series.min(), data_series.max(), 200)
                kde_values = kde(x_range)
                
                fig.add_trace(
                    go.Scatter(
                        x=x_range,
                        y=kde_values,
                        mode='lines',
                        name='KDE',
                        line=dict(color='darkblue', width=2),
                        hovertemplate='Valeur: %{x:.2f}<br>Densité: %{y:.4f}<extra></extra>',
                        showlegend=False
                    ),
                    row=1, col=2
                )
                
                # Obtenir la hauteur max du KDE pour ajuster les lignes verticales
                y_max = max(kde_values) * 1.1
            else:
                y_max = 1
            
            # Lignes verticales pour moyenne et médiane
            fig.add_trace(
                go.Scatter(
                    x=[moyenne, moyenne],
                    y=[0, y_max],
                    mode='lines',
                    name=f'Moyenne',
                    line=dict(color='red', dash='dash', width=2),
                    showlegend=False,
                    hovertemplate=f'Moyenne: {moyenne:.2f}<extra></extra>'
                ),
                row=1, col=2
            )
            
            fig.add_trace(
                go.Scatter(
                    x=[mediane, mediane],
                    y=[0, y_max],
                    mode='lines',
                    name=f'Médiane',
                    line=dict(color='violet', dash='dash', width=2),
                    showlegend=False,
                    hovertemplate=f'Médiane: {mediane:.2f}<extra></extra>'
                ),
                row=1, col=2
            )
            
            # Lignes verticales pour percentiles 25 et 75
            fig.add_trace(
                go.Scatter(
                    x=[percentile_25, percentile_25],
                    y=[0, y_max],
                    mode='lines',
                    name=f'P25',
                    line=dict(color='orange', dash='dot', width=1.5),
                    showlegend=False,
                    hovertemplate=f'Percentile 25: {percentile_25:.2f}<extra></extra>'
                ),
                row=1, col=2
            )
            
            fig.add_trace(
                go.Scatter(
                    x=[percentile_75, percentile_75],
                    y=[0, y_max],
                    mode='lines',
                    name=f'P75',
                    line=dict(color='brown', dash='dot', width=1.5),
                    showlegend=False,
                    hovertemplate=f'Percentile 75: {percentile_75:.2f}<extra></extra>'
                ),
                row=1, col=2
            )
            
            # === Ajout des statistiques dans la légende ===
            # Créer des traces invisibles pour afficher les stats dans la légende
            stats_text = [
                f"<b>Statistiques {tag_name}</b>",
                f"Nombre de valeurs: {nb_valeurs}",
                f"Moyenne: {moyenne:.2f}",
                f"Médiane: {mediane:.2f}",
                f"Écart-type: {ecart_type:.2f}",
                f"Min: {val_min:.2f}",
                f"Max: {val_max:.2f}",
                f"Percentile 25: {percentile_25:.2f}",
                f"Percentile 75: {percentile_75:.2f}"
            ]
            
            # Ajouter les statistiques comme traces invisibles pour la légende
            for i, stat in enumerate(stats_text):
                fig.add_trace(
                    go.Scatter(
                        x=[None],
                        y=[None],
                        mode='markers',
                        marker=dict(size=0),
                        name=stat,
                        showlegend=True,
                        hoverinfo='none'
                    ),
                    row=1, col=1
                )
            
            # Mise en page
            fig.update_layout(
                title_text=f"Analyse de {tag_name}",
                height=600,
                width=None,
                showlegend=True,
                hovermode='closest',
                legend=dict(
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=1.01,
                    bgcolor="rgba(255, 255, 255, 0.8)",
                    bordercolor="gray",
                    borderwidth=1
                )
            )
            
            fig.update_xaxes(title_text="Date/Temps", row=1, col=1)
            fig.update_yaxes(title_text="Valeur", row=1, col=1)
            fig.update_xaxes(title_text="Valeur", row=1, col=2)
            fig.update_yaxes(title_text="Densité", row=1, col=2)
            
            fig.show()
    
    def plot_all_selected(self):
        """
        Affiche les graphiques pour tous les tags sélectionnés.
        Utilise les noms finaux (renommés) des colonnes.
        """
        if self.data is None:
            print("Erreur : Aucune donnée disponible. Exécutez merge() d'abord.")
            return
        
        # Récupérer les noms finaux des tags sélectionnés
        tags_finaux = [item['nom'] for item in self.tags_selected]
        
        print(f"Affichage des graphiques pour {len(tags_finaux)} tags sélectionnés...")
        
        for tag in tags_finaux:
            if tag in self.data.columns:
                self.plot_tag(tag)
            else:
                print(f"Attention : tag '{tag}' non trouvé dans data")
    
    def reset_data(self):
        """
        Réinitialise self.data à partir de self.df avec renommage.
        """
        if self.df is not None:
            self.data = self._rename_columns(self.df.copy())
            print("self.data réinitialisé à partir de self.df (avec renommage)")
        else:
            print("Aucune donnée brute disponible (self.df est None)")
    
    def get_column_mapping(self):
        """
        Retourne le mapping des noms de colonnes (API -> Final).
        
        Returns:
        dict : dictionnaire de mapping
        """
        return self.rename_mapping.copy()
    
    def get_reverse_mapping(self):
        """
        Retourne le mapping inversé (Final -> API).
        
        Returns:
        dict : dictionnaire de mapping inversé
        """
        return {v: k for k, v in self.rename_mapping.items()}
    
    # Setters
    def set_url_base(self, url_base):
        """Modifie l'URL de base."""
        self.url_base = url_base
        print(f"URL de base modifiée : {url_base}")
    
    def set_tags_other(self, tags_other):
        """Modifie la liste des tags autres."""
        self.tags_other = tags_other
        print(f"Tags autres modifiés : {tags_other}")
    
    def set_tags_selected(self, tags_selected):
        """
        Modifie la liste des tags sélectionnés et met à jour le mapping.
        
        Parameters:
        tags_selected : list of dict, format [{'tag': 'nom_api', 'nom': 'nom_final'}, ...]
        """
        self.tags_selected = tags_selected
        self.rename_mapping = {item['tag']: item['nom'] for item in tags_selected}
        print(f"Tags sélectionnés modifiés : {tags_selected}")
        print(f"Nouveau mapping : {self.rename_mapping}")
    
    def set_start(self, start):
        """Modifie la date de début."""
        self.start = start
        print(f"Date de début modifiée : {start}")
    
    def set_end(self, end):
        """Modifie la date de fin."""
        self.end = end
        print(f"Date de fin modifiée : {end}")
    
    def set_interval(self, interval):
        """Modifie l'intervalle d'agrégation."""
        self.interval = interval
        print(f"Intervalle modifié : {interval}")
    
    def set_hS(self, hS):
        """Modifie l'heure de début."""
        self.hS = hS
        print(f"Heure de début modifiée : {hS}")
    
    def set_hF(self, hF):
        """Modifie l'heure de fin."""
        self.hF = hF
        print(f"Heure de fin modifiée : {hF}")
    
    def info(self):
        """
        Affiche tous les paramètres de la classe.
        """
        print("="*60)
        print("INFORMATIONS DataProcessor")
        print("="*60)
        print(f"URL de base       : {self.url_base}")
        print(f"Tags autres       : {self.tags_other}")
        print(f"Tags sélectionnés : {self.tags_selected}")
        tags_api = [item['tag'] for item in self.tags_selected]
        print(f"Tous les tags API : {self.tags_other + tags_api}")
        print(f"Mapping renommage : {self.rename_mapping}")
        print(f"Date de début     : {self.start}")
        print(f"Date de fin       : {self.end}")
        print(f"Intervalle        : {self.interval}")
        print(f"Heure de début    : {self.hS}")
        print(f"Heure de fin      : {self.hF}")
        print("-"*60)
        if self.df is not None:
            print(f"DataFrame brut (df)      : {len(self.df)} lignes, {len(self.df.columns)} colonnes")
            print(f"Colonnes df (noms API)   : {list(self.df.columns)}")
        else:
            print("DataFrame brut (df)      : Non chargé")
        
        if self.data is not None:
            print(f"DataFrame travail (data)    : {len(self.data)} lignes, {len(self.data.columns)} colonnes")
            print(f"Colonnes data (noms finaux) : {list(self.data.columns)}")
        else:
            print("DataFrame travail (data)    : Non chargé")
        print("="*60)

In [2]:
tags = [ {'tag':'CTY_A1000M_Poids container', 'nom':'A1000M_Poids' },
         {'tag' : 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'nom':'A1000M_UV_Auto' },
         {'tag' :'CTY_A1000M_Titre VA AC', 'nom':'A1000M_Labo' },
         {'tag' :'CTY_A1000M_Numéro Container', 'nom':'A1000M_Num' },
         {'tag' :'CTY_A1000R_Poids container', 'nom':'A1000R_Poids' },
         {'tag' :'CTY_A1000R_Teneur arr. Vit. A (UV)', 'nom':'A1000R_UV_Auto' },
         {'tag' :'CTY_A1000R_Titre VA AC', 'nom':'A1000R_Labo' },
         {'tag' :'CTY_A1000R_Numéro Container', 'nom':'A1000R_Num' } ]

In [3]:
# Initialisation
processor = DataProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    tags_other = [],
    tags_selected = tags,
    start='2025-01-01',
    end='2025-12-18'
)

# Afficher les infos
#processor.info()

# Charger les données
processor.merge()

# Afficher les infos après chargement
processor.info()

# Appliquer des filtres (chaînage possible)
processor.filtering(
    tag=['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo'],
    min_val=[700, 800_000, 700_000],
    max_val=[1300, 1_300_000, 1_300_000],
    na='A1000M_Poids'
)

# Ajouter des colonnes
processor.ajoute_cumul('A1000M_Poids', 'A1000M_UV_Auto', 1000, 'Cumul_UV')
processor.ajoute_cumul('A1000M_Poids', 'A1000M_Labo', 1000, 'Cumul_Labo')
processor.ajouter_moyennes_glissantes('A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Moyenne_UV', window=10)
processor.ajouter_moyennes_glissantes('A1000M_Poids', 'A1000M_Labo', 'A1000M_Moyenne_Labo', window=10)

# Modifier des paramètres
#processor.set_start('2020-01-01')
#processor.set_interval('PT30M')

# Recharger avec les nouveaux paramètres
#processor.read()

# Réinitialiser data à partir de df
#processor.reset_data()

# Afficher les infos finales
processor.info()

Données chargées : 25344 lignes, 8 colonnes
Colonnes renommées : ['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo', 'A1000M_Num', 'A1000R_Poids', 'A1000R_UV_Auto', 'A1000R_Labo', 'A1000R_Num']
INFORMATIONS DataProcessor
URL de base       : https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?
Tags autres       : []
Tags sélectionnés : [{'tag': 'CTY_A1000M_Poids container', 'nom': 'A1000M_Poids'}, {'tag': 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'nom': 'A1000M_UV_Auto'}, {'tag': 'CTY_A1000M_Titre VA AC', 'nom': 'A1000M_Labo'}, {'tag': 'CTY_A1000M_Numéro Container', 'nom': 'A1000M_Num'}, {'tag': 'CTY_A1000R_Poids container', 'nom': 'A1000R_Poids'}, {'tag': 'CTY_A1000R_Teneur arr. Vit. A (UV)', 'nom': 'A1000R_UV_Auto'}, {'tag': 'CTY_A1000R_Titre VA AC', 'nom': 'A1000R_Labo'}, {'tag': 'CTY_A1000R_Numéro Container', 'nom': 'A1000R_Num'}]
Tous les tags API : ['CTY_A1000M_Poids container', 'CTY_A1000M_Teneur arr. Vit. A (UV)', 'CTY_A1000M_Titre VA AC', 'CTY_A1000M_Numéro Contai

In [6]:
processor.data.head(20)

,A1000M_Poids,A1000M_UV_Auto,A1000M_Labo,A1000M_Num,A1000R_Poids,A1000R_UV_Auto,A1000R_Labo,A1000R_Num,Cumul_UV,Cumul_Labo,A1000M_Moyenne_UV,A1000M_Moyenne_Labo
timestamp,,,,,,,,,,,,
2025-01-01 10:00:00+00:00,1055.666667,1.039223e+06,1.041903e+06,53.0,NaN,1029000.0,1037800.0,NaN,1.097073e+06,1.099902e+06,NaN,NaN
2025-01-01 19:00:00+00:00,1030.000000,1.039000e+06,1.038726e+06,95.0,NaN,1029000.0,1037800.0,NaN,1.070170e+06,1.069887e+06,NaN,NaN
2025-01-02 07:00:00+00:00,1144.000000,1.039000e+06,1.030161e+06,60.0,NaN,1029000.0,1037800.0,NaN,1.188616e+06,1.178504e+06,NaN,NaN
2025-01-02 12:00:00+00:00,967.000000,1.039000e+06,1.038700e+06,2.0,NaN,1029000.0,1037800.0,NaN,1.004713e+06,1.004423e+06,NaN,NaN
2025-01-02 15:00:00+00:00,935.000000,1.039000e+06,1.009350e+06,19.0,NaN,1029000.0,1037800.0,NaN,9.714650e+05,9.437419e+05,NaN,NaN
2025-01-02 19:00:00+00:00,1061.000000,1.039000e+06,1.012976e+06,45.0,NaN,1029000.0,1037800.0,NaN,1.102379e+06,1.074768e+06,NaN,NaN
2025-01-02 22:00:00+00:00,1040.000000,1.039000e+06,1.013700e+06,NaN,NaN,1029000.0,1037800.0,NaN,1.080560e+06,1.054248e+06,NaN,NaN
2025-01-03 00:00:00+00:00,1099.000000,1.039000e+06,1.031300e+06,7.0,NaN,1029000.0,1037800.0,NaN,1.141861e+06,1.133399e+06,NaN,NaN
2025-01-03 03:00:00+00:00,1047.000000,1.039000e+06,1.031300e+06,97.0,NaN,1029000.0,1037800.0,NaN,1.087833e+06,1.079771e+06,NaN,NaN


In [5]:
processor.plot_tag(tag=['A1000M_Poids','A1000M_UV_Auto'])